In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import requests
import json
import numpy as np
import pickle
from collections import OrderedDict
import os
import asyncio
import itertools
from copy import deepcopy
from tqdm import tqdm
import ast

from openai import AsyncOpenAI, OpenAI
import tiktoken

try:
    from .gog_prompts import PROMPTS
except ImportError:
    from gog_prompts import PROMPTS

In [ ]:
class GOGNode:
    def __init__(self, name, desc, tools, mats, postconditions, name_emb, tools_emb, mats_emb, postconditions_emb):
        self.name = name
        self.desc = desc
        self.aliases = []
        self.tools = [tools]
        self.mats = [mats]
        self.postconditions = [postconditions]
        self.name_emb = name_emb

        # These are lists of embeddings
        self.tools_emb = [tools_emb]
        self.mats_emb = [mats_emb]
        self.postconditions_emb = [postconditions_emb]

    def add_alternative(self, tools, mats, postconditions, tools_emb, mats_emb, postconditions_emb):
        self.tools.append(tools)
        self.mats.append(mats)
        self.postconditions.append(postconditions)
        self.tools_emb.append(tools_emb)
        self.mats_emb.append(mats_emb)
        self.postconditions_emb.append(postconditions_emb)

class GOGEdge:
    def __init__(self, goal, subgoal, description):
        self.goal = goal
        self.subgoal = subgoal
        self.description = description

In [ ]:
class GOGKB:
    def __init__(self, url="", port="", llm="", embedding_model=""):
        self.nodes = []
        self.edges = []
        self.goals_emb = []

        self.name2node = {}
        self.emb2node = {}
        # Node to Edges
        self.goal2subgoals = {}

        # Node name to postconditions of potential subgoal
        # {<name>: {"node": <node>, "tools": [<tools>], "mats": [<mats>]}}
        self.name2unmatchedreqs = {}
        self.postconditions2nodes = {}

        self.set_llm(url, port, llm)
        self.set_embedding(url, port, embedding_model)

        self.doc_dir = "../../../data/final_docs/"
        self.kb_dir = "gog_graph/"
        self.kb_file = self.kb_dir + "kb.pkl"

        # Run async calls if using openai
        self.max_async = 16

        # The cosine similarity threshold for considering two goals to be equivalent
        # Name and postconditions of the goal are checked
        self.goal_similarity_threshold = 0.92

    @classmethod
    def load_kb(cls, kb_file="./gog_graph/kb.pkl"):
        with open(kb_file, "rb") as f:
            kb = pickle.load(f)
        return kb

    def set_llm(self, url="", port="", llm=""):
        self.url = url
        self.port = port
        self.chat_endpoint = "/v1"
        self.llm = llm
        if url != "":
            self.chat_url = self.url + ":" + self.port + self.chat_endpoint
        else:
            self.chat_url = ""

    def set_embedding(self, url="", port="", embedding_model=""):
        self.url = url
        self.port = port
        self.embed_endpoint = "/v1"
        self.embedding_model = embedding_model
        if url != "":
            self.embed_url = self.url + ":" + self.port + self.embed_endpoint
        else:
            self.embed_url = ""

    def save_kb(self):
        with open(self.kb_file, "wb") as f:
            pickle.dump(self, f)

    def add_node(self, node, name_embedding, postconditions):
        self.nodes.append(node)
        self.name2node[node.name] = node
        self.emb2node[tuple(name_embedding)] = node
        self.goals_emb.append(name_embedding)

        if postconditions != "None":
            post = list(postconditions.keys())[0]
            if post not in self.postconditions2nodes:
                self.postconditions2nodes[post] = []    
            self.postconditions2nodes[post].append(node)

    def add_edge(self, goal, subgoal, relation_desc):
        # Check if relation already exists
        if goal in self.goal2subgoals:
            exists = False
            for e in self.goal2subgoals[goal]:
                if e.subgoal.name == subgoal.name:
                    exists = True
                    break
            if exists:
                return
        else:
            self.goal2subgoals[goal] = []

        edge = GOGEdge(goal, subgoal, relation_desc)
        self.edges.append(edge)
        self.goal2subgoals[goal].append(edge)

    async def gather_batch_embed_text(self, queries):
        client = AsyncOpenAI()
        tasks = [self.embed_text_async(q, client) for q in queries]
        results = await asyncio.gather(*tasks)
        return results

    async def embed_text_async(self, text, client):
        if not isinstance(text, list):
            text = [text]
        response = await client.embeddings.create(input=text, model=self.embedding_model, encoding_format="float")
        if len(response.data) == 1:
            data = response.data[0].embedding
        else:
            data = [x.embedding for x in response.data]

        return data

    async def gather_batch_llm_query(self, queries):
        async with AsyncOpenAI() as client:
            tasks = [self.query_llm_async(q, client) for q in queries]
            results = await asyncio.gather(*tasks)
        return results

    async def query_llm_async(self, query, client):
        response = await client.chat.completions.create(
            model= self.llm,
            temperature= 0.0,
            messages=[
                {
                    "role": "user",
                    "content": query,
                }
            ]
        )

        return response.choices[0].message.content

    def query_llm(self, query):
        if self.chat_url != "":
            client = OpenAI(base_url=self.chat_url, api_key="123")
        else:
            client = OpenAI()
        response = client.chat.completions.create(
            model= self.llm,
            temperature= 0.0,
            messages=[
                {
                    "role": "user",
                    "content": query,
                }
            ]
        )
        output = response.choices[0].message.content

        return output

    def get_reqs(self, tools, mats):
        reqs = []

        if tools != "None":
            tools_json = tools
        else:
            tools_json = {}

        if mats != "None":
            mats_json = mats
        else:
            mats_json = {}

        for tool in tools_json:
            if tool == "fuel":
                reqs.append("logs")
            elif tool == "None":
                continue
            else:
                reqs.append(tool)

        for mat in mats_json:
            if mat == "None":
                continue
            reqs.append(mat)

        return reqs

    def check_goal_names_equivalent(self, goal_a, goal_b):
        goal_a_split = goal_a.split(" ")
        goal_b_split = goal_b.split(" ")

        if goal_a_split[0] == goal_b_split[0]:
            if goal_a_split[1] == "a":
                goal_a_item = " ".join(goal_a_split[2:])
            else:
                goal_a_item = " ".join(goal_a_split[1:])

            if goal_b_split[1] == "a":
                goal_b_item = " ".join(goal_b_split[2:])
            else:
                goal_b_item = " ".join(goal_b_split[1:])

            sim = cosine_similarity([self.embed_text(goal_a_item)],[self.embed_text(goal_b_item)])[0][0]

            if sim > self.goal_similarity_threshold:
                return True

        return False

    def check_conditions_equivalent(self, conds_a, conds_b):
        if conds_a == conds_b:
            return True

        if isinstance(conds_a,str) or isinstance(conds_b,str):
            return False

        if len(conds_a) != len(conds_b):
            return False

        keys_a = sorted(list(conds_a.keys()))
        keys_b = sorted(list(conds_b.keys()))
        for a,b in zip(keys_a, keys_b):
            sim = cosine_similarity([self.embed_text(a)],[self.embed_text(b)])[0][0]
            if sim < self.goal_similarity_threshold:
                return False
            if not conds_a[a] == conds_b[b]:
                return False

        return True

    def build_kb(self):
        doc_dir_files = [x for x in os.listdir(self.doc_dir) if ".json" in x]

        goal_extraction_prompt = PROMPTS["goal_extraction"]
        goal_merge_prompt = PROMPTS["goal_merge"]
        subgoal_merge_prompt = PROMPTS["subgoal_merge_and_expand"]
        tuple_delimiter = PROMPTS["tuple_delimiter"]
        record_delimiter = PROMPTS["record_delimiter"]
        completion_delimiter = PROMPTS["completion_delimiter"]

        encoding = tiktoken.encoding_for_model("gpt-4o-mini")

        goal_list_all = ""
        subgoal_list_all = ""
        raw_llm_output = ""
        reduced_goal_list_all = ""
        reduced_subgoal_list_all = ""

        async_docs = self.max_async if self.use_openai_llm else 1
        pbar = tqdm(range(0, len(doc_dir_files), async_docs))
        for idx in pbar:
            files = doc_dir_files[idx:idx+async_docs]

            # {<name>: {"node": <node>, "reqs": [<tools and mats>]}}
            added_goals = {}

            # {<original_name>: <updated_name>}
            renamed_goals = {}

            doc_texts = []
            for file in files:
                with open(self.doc_dir+file, "r") as f:
                    doc_text = f.read()
                    doc_texts.append(doc_text)

            extracted_goal_list = []
            extracted_subgoal_list = []
            goal_list_str = ""
            subgoal_list_str = ""

            output = self.query_llm(goal_extraction_prompt.format(tuple_delimiter=tuple_delimiter, record_delimiter=record_delimiter, completion_delimiter=completion_delimiter, input_text=doc_chunks[i]))
            
            goal_tuple_strs = output.replace("\n", "").replace(completion_delimiter, record_delimiter).split(record_delimiter)
            last_reason = ""
            for g in goal_tuple_strs:
                if g == "" or g.isspace() or tuple_delimiter not in g:
                    continue

                entry_type = g[1:g.index(tuple_delimiter)].replace("\"", "")
                entry = g.replace(tuple_delimiter, ",")
                if entry_type == "goal":
                    name = g.split(tuple_delimiter)[1]
                    goal_list_str += entry + "\n"
                elif entry_type == "subgoal":
                    subgoal_list_str += entry + "\n"

            if len(goal_list_str) == 0:
                continue

            goal_tuple_strs = goal_list_str.split("\n")
            goal_tuple_strs += subgoal_list_str.split("\n")

            goal_tuples = []
            to_embed = []
            extracted_subgoals = []
            for g in goal_tuple_strs:
                if g == "" or g.isspace():
                    continue
                try:
                    goal_tuple = ast.literal_eval(g)
                except Exception as e:
                    print(f'Exception: {e}')
                    continue

                entry_type = goal_tuple[0]
                if entry_type == "goal":
                    if len(goal_tuple) < 6:
                        continue
                    name = goal_tuple[1]
                    desc = goal_tuple[2]
                    tools = goal_tuple[3] if goal_tuple[3] is not None else "None"
                    mats = goal_tuple[4] if goal_tuple[4] is not None else "None"
                    postconditions = goal_tuple[5] if goal_tuple[5] is not None else "None"
                    if tools == {"None"} or tools == {}:
                        tools = "None"

                    if isinstance(tools, dict):
                        if len(tools) == 0:
                            tools = "None"
                            tools_str = "None"
                        else:
                            tools_str = ", ".join(sorted(list(tools.keys())))
                    else:
                        # Incorrectly formatted goal
                        if tools.lower() != "none":
                            continue
                        tools_str = "None"

                    if mats == {"None"} or mats == {}:
                        mats = "None"
                    if isinstance(mats, dict):
                        to_change = []
                        for m in mats:
                            if m in name_normalize or m.lower() == "none":
                                to_change.append(m)
                        for m in to_change:
                            if m.lower() == "none":
                                del mats[m]
                                continue
                            mats[name_normalize[m]] = mats[m]
                            del mats[m]
                        mats_str = ", ".join(sorted(list(mats.keys())))
                    else:
                        # Incorrectly formatted goal
                        if mats.lower() != "none":
                            continue
                        mats_str = "None"

                    if postconditions == {"None"} or postconditions == {}:
                        postconditions = "None"
                    if isinstance(postconditions, dict):
                        to_change = []
                        for p in postconditions:
                            if p in name_normalize:
                                to_change.append(p)
                        for p in to_change:
                            postconditions[name_normalize[p]] = postconditions[p]
                            del postconditions[p]
                        postconditions_str = ", ".join(sorted(list(postconditions.keys())))
                    else:
                        # Incorrectly formatted goal
                        if postconditions.lower() != "none":
                            continue
                        postconditions_str = "None"

                    goal_tuples.append((name, desc, tools, mats, postconditions))
                    to_embed.append([name, tools_str, mats_str, postconditions_str])
                elif entry_type == "subgoal":
                    extracted_subgoals.append((goal_tuple[1], goal_tuple[2], goal_tuple[3]))

            embedded_nodes = []
            for e in to_embed:
                result = []
                for text in e:
                    result.append(self.embed_text(text))
                embedded_nodes.append(result)

            # Use heuristic methods to merge into existing goal DB
            for goal_tuple, embedded_node in zip(goal_tuples, embedded_nodes):
                name = goal_tuple[0]
                desc = goal_tuple[1]
                tools = goal_tuple[2]
                mats = goal_tuple[3]
                postconditions = goal_tuple[4]

                name_emb = embedded_node[0]
                tools_emb = embedded_node[1]
                mats_emb = embedded_node[2]
                postconditions_emb = embedded_node[3]

                if len(self.nodes) == 0:
                    node = GOGNode(name, desc, tools, mats, postconditions, name_emb, tools_emb, mats_emb, postconditions_emb)
                    self.add_node(node, name_emb, postconditions)
                    reqs = self.get_reqs(tools, mats)
                    added_goals[name] = {"node": node, "reqs": reqs}
                    continue

                top_goal, goal_name_sim = self.query_goals(name, top_k=1)
                top_goal = top_goal[0]
                goal_name_sim = goal_name_sim[0]
                
                # All different post-conditions should produce the same results for a given goal
                # The main difference is the number of items produced, so just check the first post-condition
                check_aliases = [top_goal.name] + top_goal.aliases
                same_goal_name = any([self.check_goal_names_equivalent(name, alias) for alias in check_aliases])
                same_tools = [self.check_conditions_equivalent(tools, top_tools) for top_tools in top_goal.tools]
                same_mats = [self.check_conditions_equivalent(mats, top_mats) for top_mats in top_goal.mats]
                same_post = [self.check_conditions_equivalent(postconditions, top_posts) for top_posts in top_goal.postconditions]
                same_conds = [all([t,m,c]) for t,m,c in zip(same_tools, same_mats, same_post)]

                if same_goal_name or any(same_conds):
                    if not same_goal_name:
                        # Check if the names are similar
                        if cosine_similarity([top_goal.name_emb], [name_emb])[0][0] < self.goal_similarity_threshold:
                            top_goal.aliases.append(name)

                    if not any(same_conds):#exists:
                        top_goal.add_alternative(tools, mats, postconditions, tools_emb, mats_emb, postconditions_emb)
                        if isinstance(postconditions, dict):
                            post = list(postconditions.keys())[0]
                            if post not in self.postconditions2nodes:
                                self.postconditions2nodes[post] = [top_goal]
                        reqs = self.get_reqs(tools, mats)
                        if len(reqs) > 0:
                            added_goals[name] = {"node": top_goal, "reqs": reqs}

                    renamed_goals[name] = top_goal.name
                else:
                    node = GOGNode(name, desc, tools, mats, postconditions, name_emb, tools_emb, mats_emb, postconditions_emb)
                    self.add_node(node, name_emb, postconditions)
                    reqs = self.get_reqs(tools, mats)
                    if len(reqs) > 0:
                        added_goals[name] = {"node": node, "reqs": reqs}

            # Subgoal Merge
            # Check which subgoals are valid and which can be merged, if any
            for s in extracted_subgoals:
                goal_name = s[0]
                subgoal_name = s[1]
                relation_desc = s[2]
                if goal_name in renamed_goals:
                    goal_name = renamed_goals[goal_name]

                if subgoal_name in renamed_goals:
                    subgoal_name = renamed_goals[subgoal_name]

                if goal_name not in self.name2node or subgoal_name not in self.name2node:
                    continue

                goal = self.name2node[goal_name]
                subgoal = self.name2node[subgoal_name]

                subgoal_output = subgoal.postconditions[0]
                if subgoal_output == "None":
                    continue
                subgoal_output = list(subgoal_output.keys())[0]

                # Check if the goal pre-reqs match the subgoal post-conditions
                goal_reqs = []
                for tools, mats in zip(goal.tools, goal.mats):
                    goal_reqs += [x for x in self.get_reqs(tools, mats) if x not in goal_reqs]

                if subgoal_output not in goal_reqs:
                    continue

                self.add_edge(goal, subgoal, relation_desc)

                if goal_name in added_goals:
                    if subgoal_output in added_goals[goal_name]["reqs"]:
                        added_goals[goal_name]["reqs"].remove(subgoal_output)

                        if added_goals[goal_name]["reqs"] == []:
                            added_goals.pop(goal_name)

            # Subgoal Inference
            # After new goals have been added, determine whether we can add relationships between new and existing goals
            # Check if there are any unmatched pre-conditions
            for g in added_goals:
                for r in added_goals[g]["reqs"]:
                    if r in self.postconditions2nodes:
                        for subgoal in self.postconditions2nodes[r]:
                            goal = added_goals[g]["node"]
                            # For now, use a default description
                            relation_desc = subgoal.name + " is used by " + goal.name

                            self.add_edge(goal, subgoal, relation_desc)

            # Update existing goals to link with new goals
            for g in self.name2node:
                existing_goal = self.name2node[g]
                for exist_tools, exist_mats in zip(existing_goal.tools, existing_goal.mats):
                    reqs = self.get_reqs(exist_tools, exist_mats)
                    for new_g in added_goals:
                        new_goal = added_goals[new_g]["node"]
                        post = new_goal.postconditions[0]
                        if post == "None":
                            continue
                        post = list(post.keys())[0]
                        # Add a subgoal relation if it doesn't exist
                        if post in reqs:
                            self.add_edge(existing_goal, new_goal, new_goal.name + " is used by " + existing_goal.name)

            completed_files += files
            # Save after each file just in case
            self.save_kb()

            with open(completed_files_txt, "w") as f:
                f.write(", ".join(completed_files))

    def load_recipes(self, files, recipe_dir):
        recipe_str = ""
        for file in files:
            recipe_str += "--- Recipe: {name} ---".format(name=file) + "\n"
            with open(recipe_dir+file) as f:
                recipe = json.load(f)
            recipe_str += json.dumps(recipe) + "\n"
        return recipe_str

    def process_doc(self, file, doc_dir):
        with open(doc_dir+file) as f:
            data_json = json.load(f)

        # Get table data
        tables = []
        for t in data_json["tables"]:
            row_len = math.gcd(len(t["headers"]["text"]), len(t["cells"]["text"]))
            table = "--- Table Start ---\n"
            table += "Headers: "
            for i, h in enumerate(t["headers"]["text"]):
                table += h
                if i < len(t["headers"]["text"])-1:
                    table += ", "
            table += "\n"
            table += "Cells: \n"
            for i, c in enumerate(t["cells"]["text"]):
                table += c
                if (i+1) % row_len == 0:
                    table += "\n"
                elif i < len(t["cells"]["text"])-1:
                    table += ", "
            table += "--- Table End ---\n"
            tables.append({"table": table, "bbox": t["bbox"]})

        # Arrange data in order from top to bottom
        doc = ""
        i = 0
        j = 0
        while i < len(data_json["texts"]) or j < len(tables):
            if j >= len(tables) or (i < len(data_json["texts"]) and data_json["texts"][i]["bbox"][1] < tables[j]["bbox"][1]):
                doc += data_json["texts"][i]["text"]
                i += 1
            else:
                doc += tables[j]["table"]
                j += 1
            doc += "\n"
        return doc

    def embed_text(self, text):
        if self.embed_url != "":
            client = OpenAI(base_url=self.embed_url, api_key="123")
        else:
            client = OpenAI()
        if not isinstance(text, list):
            text = [text]
        response = client.embeddings.create(input=text, model=self.embedding_model, encoding_format="float")
        if len(response.data) == 1:
            data = response.data[0].embedding
        else:
            data = [x.embedding for x in response.data]

        return data

    # Return the top-k nodes that have goal names most similar to the query
    def query_goals(self, query, top_k=3):
        embed_query = self.embed_text(query)
        similarities = [x[0] for x in cosine_similarity(self.goals_emb, [embed_query])]

        # Get top-k similarities (unordered)
        sim_idxs = [i for i in np.argpartition(similarities, -top_k)[-top_k:]]
        sim_values = [similarities[i] for i in sim_idxs]
        top = [self.goals_emb[i] for i in sim_idxs]

        nodes = []
        for emb in top:
            nodes.append(self.emb2node[tuple(emb)])

        return nodes, sim_values

    # Returns a list of alternative trees that can achieve the given goal
    # Assume that there's only one way to craft tools
    def dfs(self, goal, current_path=None):
        if type(goal) is str:
            goal = self.name2node[goal]
        if current_path is None:
            current_path = set()
        elif goal.name in current_path:
            return []
        current_path.add(goal.name)

        trees = []
        for tools, mats, post in zip(goal.tools, goal.mats, goal.postconditions):
            mat_paths = []
            tool_subgoals = []
            
            if tools != "None":
                for t in tools:
                    if t in self.postconditions2nodes:
                        tool_subgoals += self.postconditions2nodes[t]

            if mats != "None":
                for m in mats:
                    if m in self.postconditions2nodes:
                        mat_paths.append(self.postconditions2nodes[m])
                mat_combinations = list(itertools.product(*mat_paths))
            else:
                mat_combinations = ["None"]

            for idx, mat_subgoals in enumerate(mat_combinations):
                goal_info = {
                    "description": goal.desc,
                    "aliases": goal.aliases,
                    "tools": tools,
                    "materials": mats,
                    "postconditions": post,
                    "subgoals": [],
                }

                subgoals = list(mat_subgoals) + tool_subgoals
                if goal in self.goal2subgoals:
                    subgoal_edges = [x for x in self.goal2subgoals[goal] if x.subgoal in subgoals]
                else:
                    return [{goal.name: goal_info}]

                subgoal_trees = []
                for e in subgoal_edges:
                    subgoal = e.subgoal

                    goal_info["subgoals"].append({
                        "subgoal": subgoal.name,
                        "relationship_description": e.description
                    })
                    
                    results = self.dfs(subgoal, current_path.copy())
                    subgoal_trees.append(results)

                subgoal_combinations = list(itertools.product(*subgoal_trees))
                for sub_comb in subgoal_combinations:
                    tree = {}
                    tree[goal.name] = goal_info
                    # Check if a goal already exists in the tree in the loop below
                    # If yes, there's a conflict (do not need to consider two ways of achieving the same goal in one alternative, e.g. using planks to craft some sticks and bamboo to craft others)
                    # Not just if a goal already exists, but if there is a way to obtain the post-conditions of a given goal already in the tree
                    skip = False
                    for s in sub_comb:
                        for k in s:
                            # This checks if the materials used are the same if the goal already exists
                            if k in tree and tree[k] != s[k]:
                                skip = True
                                break

                            # Check the post-conditions
                            elif k not in tree:
                                for t in tree:
                                    if tree[t]["postconditions"].keys() == s[k]["postconditions"].keys():
                                        skip = True
                                        break
                            if skip:
                                break        

                        if skip:
                            break

                        tree.update(s)
                    if not skip:
                        trees.append(tree)
        return trees

In [ ]:
if __name__ == "__main__":
    from tqdm import tqdm
    import re
    import os
    import ast
    import math

    kb = GOGKB()

    kb.build_kb()
    kb.save_kb()